# Practica Agentes Inteligentes — Agente de Peliculas

Notebook completo de la practica. Recopila todos los modulos del proyecto en un unico documento ejecutable por celdas.

**Componentes incluidos:**
1. `config.py` — Configuracion centralizada
2. `requirements.txt` — Dependencias
3. `movie_scraper.py` — Scraper de SensaCine (web scraping con BeautifulSoup)
4. `cartelera_scraper.py` — Scraper de cartelera de Madrid (eCartelera) + integracion SensaCine
5. `user_profile.json` — Perfil de filtrado de usuario
6. `telegram_bot.py` — Bot de Telegram con LLM open source (Ollama)
7. `web_app.py` — Interfaz web Flask
8. `alexa_lambda.py` — Skill de Alexa (AWS Lambda)
9. `alexa_interaction_model.json` — Modelo de interaccion de la skill
10. `cron_cartelera.sh` — Automatizacion semanal por cron

Cada celda esta etiquetada con el fichero original al que pertenece.

## 1. Instalacion de dependencias

Contenido de `requirements.txt`.

In [ ]:
# requirements.txt
%pip install \
    "requests>=2.31.0" \
    "beautifulsoup4>=4.12.0" \
    "lxml>=5.0.0" \
    "flask>=3.0.0" \
    "python-telegram-bot>=21.0" \
    "ask-sdk-core>=1.19.0" \
    "ollama>=0.4.0"

## 2. Configuracion centralizada (`config.py`)

Definimos todas las variables de configuracion como un modulo `config` cargado en memoria, de forma que el resto de celdas puedan importarlo igual que los scripts originales.

In [ ]:
# config.py
import sys, types

config = types.ModuleType("config")

# --- Telegram ---
config.TELEGRAM_BOT_TOKEN = "8681004744:AAH-t0sPHD5Zr_2lMXpoKIYNXzE7n5U7YAY"  # Obtener de @BotFather
config.TELEGRAM_CHAT_ID = "6451572961"  # Chat ID para envio automatico

# --- LLM Open Source (Ollama) ---
config.OLLAMA_URL = "http://localhost:11434"
config.OLLAMA_MODEL = "qwen2.5:3b"

# --- Alternativa: Groq API ---
config.GROQ_API_KEY = ""
config.GROQ_MODEL = "llama-3.3-70b-versatile"

# --- Scraping ---
config.IMDB_BASE_URL = "https://www.imdb.com"
config.ECARTELERA_URL = "https://www.ecartelera.com"
config.REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# --- Cache ---
config.CACHE_FILE = "movie_cache.json"

# --- Perfil de usuario ---
config.USER_PROFILE_FILE = "user_profile.json"

# --- Web App ---
config.FLASK_HOST = "0.0.0.0"
config.FLASK_PORT = 5000
config.FLASK_DEBUG = True

# Registramos el modulo para que `import config` funcione en celdas siguientes
sys.modules["config"] = config
print("Modulo config registrado.")

## 3. Perfil de usuario (`user_profile.json`)

Definimos los generos con su nota minima y los directores favoritos. Se persisten en disco para que los usen los modulos.

In [ ]:
# user_profile.json
import json

user_profile = {
    "genres": {
        "Sci-Fi": 6.0,
        "Action": 6.5,
        "Drama": 7.0,
        "Comedy": 6.0,
        "Horror": 5.5,
        "Animation": 7.0,
        "Thriller": 6.5,
        "Biography": 7.0
    },
    "favorite_directors": [
        "Christopher Nolan",
        "Denis Villeneuve",
        "Quentin Tarantino",
        "Pedro Almod\u00f3var",
        "Martin Scorsese"
    ]
}

with open("user_profile.json", "w", encoding="utf-8") as f:
    json.dump(user_profile, f, ensure_ascii=False, indent=2)

print("Perfil guardado en user_profile.json")
print(json.dumps(user_profile, ensure_ascii=False, indent=2))

## 4. Scraper de SensaCine (`movie_scraper.py`)

Web scraping real con BeautifulSoup + requests. **No usa APIs.** Extrae titulo, nota, votos, sinopsis, director, duracion, genero, año y poster a partir del HTML de SensaCine.com.

Pasos:
1. Buscar la pelicula en `/buscar/?q=...` y extraer el ID codificado en base64 dentro de `data-entity-id`.
2. Decodificar `Movie:XXXXX` y construir la URL `/peliculas/pelicula-XXXXX/`.
3. Parsear el JSON-LD embebido + selectores HTML para datos adicionales (nota, año).

In [ ]:
# movie_scraper.py
import argparse
import base64
import json
import os
import re
import sys
import types

import requests
from bs4 import BeautifulSoup

import config

SENSACINE_BASE = "https://www.sensacine.com"

# ============================================================
# Cache
# ============================================================

def load_cache():
    """Carga la cache de peliculas desde disco."""
    if os.path.exists(config.CACHE_FILE):
        with open(config.CACHE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_cache(cache):
    """Guarda la cache de peliculas en disco."""
    with open(config.CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


# ============================================================
# PASO 1 - Buscar pelicula scrapeando la pagina de busqueda
# ============================================================

def search_movie(title):
    """Busca una pelicula en SensaCine scrapeando la pagina de resultados HTML."""
    search_url = f"{SENSACINE_BASE}/buscar/?q={requests.utils.quote(title)}"
    print(f"  [SCRAPING] GET {search_url}", file=sys.stderr)

    r = requests.get(search_url, headers=config.REQUEST_HEADERS, timeout=15)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "lxml")

    cards = soup.find_all("div", class_="entity-card")
    print(f"  [SCRAPING] Parseando HTML... {len(cards)} resultados encontrados", file=sys.stderr)

    if not cards:
        return None, None

    for card in cards:
        title_el = card.find("h2", class_="meta-title")
        card_title = title_el.get_text(strip=True) if title_el else ""

        entity_div = card.find(attrs={"data-entity-id": True})
        if not entity_div:
            continue

        encoded_id = entity_div.get("data-entity-id", "")
        if not encoded_id:
            continue

        try:
            decoded = base64.b64decode(encoded_id).decode("utf-8")
        except Exception:
            continue

        if not decoded.startswith("Movie:"):
            continue

        movie_id = decoded.split(":")[1]
        movie_path = f"/peliculas/pelicula-{movie_id}/"

        print(f"  [SCRAPING] Resultado: '{card_title}' -> ID {movie_id}", file=sys.stderr)
        return card_title, movie_path

    return None, None


# ============================================================
# PASO 2 - Scrapear la pagina de detalle
# ============================================================

def _parse_iso_duration(iso_str):
    """Convierte duracion ISO 8601 (PT02H15M00S) a formato legible (2h 15min)."""
    if not iso_str:
        return "N/A"
    match = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", iso_str)
    if not match:
        return iso_str
    hours = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    parts = []
    if hours:
        parts.append(f"{hours}h")
    if minutes:
        parts.append(f"{minutes}min")
    return " ".join(parts) if parts else "N/A"


def scrape_movie_page(movie_path):
    """Scrapea la pagina de detalle de una pelicula en SensaCine."""
    url = SENSACINE_BASE + movie_path
    print(f"  [SCRAPING] GET {url}", file=sys.stderr)

    r = requests.get(url, headers=config.REQUEST_HEADERS, timeout=15)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "lxml")
    print(f"  [SCRAPING] Parseando HTML de la ficha ({len(r.text):,} bytes)...", file=sys.stderr)

    jsonld_tag = soup.find("script", type="application/ld+json")
    ld_data = {}
    if jsonld_tag and jsonld_tag.string:
        try:
            ld_data = json.loads(jsonld_tag.string)
            print(f"  [SCRAPING] JSON-LD extraido del HTML", file=sys.stderr)
        except json.JSONDecodeError:
            pass

    titulo = ld_data.get("name", "")

    titulo_original = ld_data.get("alternateName", titulo)
    if not titulo_original or titulo_original == titulo:
        for item in soup.find_all("div", class_="meta-body-item"):
            text = item.get_text(" ", strip=True)
            if "original" in text.lower():
                parts = re.split(r"original\s*", text, flags=re.I)
                if len(parts) > 1:
                    titulo_original = parts[1].strip()
                break

    sinopsis = ld_data.get("description", "")
    if not sinopsis:
        synopsis_div = soup.find("div", class_="content-txt")
        if synopsis_div:
            sinopsis = synopsis_div.get_text(strip=True)

    directors_data = ld_data.get("director", [])
    if isinstance(directors_data, dict):
        directors_data = [directors_data]
    directores = [d.get("name", "") for d in directors_data if d.get("name")]
    if not directores:
        for item in soup.find_all("div", class_="meta-body-item"):
            text = item.get_text(" ", strip=True)
            if "dirigida por" in text.lower():
                links = item.find_all("a")
                directores = [a.get_text(strip=True) for a in links]
                break

    generos = ld_data.get("genre", [])
    if isinstance(generos, str):
        generos = [generos]

    duracion_iso = ld_data.get("duration", "")
    duracion = _parse_iso_duration(duracion_iso)
    if duracion == "N/A":
        for item in soup.find_all("div", class_="meta-body-item"):
            match = re.search(r"(\d+)\s*h\s*(\d+)\s*min", item.get_text())
            if match:
                duracion = f"{match.group(1)}h {match.group(2)}min"
                break

    poster_data = ld_data.get("image", {})
    poster = poster_data.get("url", "") if isinstance(poster_data, dict) else ""

    año = ""
    for item in soup.find_all("div", class_="meta-body-item"):
        text = item.get_text(strip=True)
        year_match = re.search(r"de\s+(\d{4})", text)
        if year_match:
            año = int(year_match.group(1))
            break

    nota = "N/A"
    votos = 0
    best_rating = "5"

    agg_rating = ld_data.get("aggregateRating", {})
    if agg_rating:
        nota_raw = str(agg_rating.get("ratingValue", "")).replace(",", ".")
        try:
            nota = round(float(nota_raw), 1)
        except ValueError:
            pass
        votos_raw = str(agg_rating.get("ratingCount", "0"))
        try:
            votos = int(votos_raw.replace(",", "").replace(".", ""))
        except ValueError:
            votos = 0
        best_rating = str(agg_rating.get("bestRating", "5"))

    if nota == "N/A":
        rating_items = soup.find_all("div", class_="rating-item-content")
        for ri in rating_items:
            note_el = ri.find("span", class_=re.compile(r"note"))
            if note_el:
                note_text = note_el.get_text(strip=True).replace(",", ".")
                try:
                    nota = round(float(note_text), 1)
                    break
                except ValueError:
                    continue

    print(f"  [SCRAPING] Datos extraidos: {titulo} ({año}) - Nota: {nota}/{best_rating}", file=sys.stderr)

    return {
        "titulo": titulo,
        "titulo_original": titulo_original,
        "año": año,
        "nota": nota,
        "nota_escala": f"/{best_rating}",
        "votos": votos,
        "sinopsis": sinopsis,
        "director": ", ".join(directores) if directores else "N/A",
        "duracion": duracion,
        "genero": ", ".join(generos) if generos else "N/A",
        "poster": poster,
        "url": url,
    }


# ============================================================
# Funcion principal reutilizable
# ============================================================

def get_movie_info(title, use_cache=True):
    """Obtiene info de pelicula por titulo. Usa cache local."""
    cache = load_cache() if use_cache else {}
    cache_key = title.lower().strip()

    if cache_key in cache:
        print(f"  [CACHE] '{title}' obtenido de cache local", file=sys.stderr)
        return cache[cache_key]

    card_title, movie_path = search_movie(title)
    if not movie_path:
        return None

    movie_info = scrape_movie_page(movie_path)
    if not movie_info:
        return None

    if use_cache:
        cache[cache_key] = movie_info
        save_cache(cache)

    return movie_info


CAMPOS_VALIDOS = ["titulo", "titulo_original", "año", "nota", "votos",
                  "sinopsis", "director", "duracion", "genero", "poster", "url"]


def format_movie_text(movie, campos=None):
    """Formatea la info de una pelicula como texto legible."""
    escala = movie.get("nota_escala", "/5")

    if campos:
        lines = []
        for c in campos:
            if c in movie:
                val = movie[c]
                if c == "nota":
                    val = f"{val}{escala}"
                lines.append(f"  {c.capitalize()}: {val}")
        return "\n".join(lines)

    lines = [
        f"  Titulo: {movie['titulo']}",
        f"  Titulo Original: {movie['titulo_original']}",
        f"  Año: {movie['año']}",
        f"  Nota SensaCine: {movie['nota']}{escala}",
        f"  Votos: {movie['votos']:,}",
        f"  Director: {movie['director']}",
        f"  Duracion: {movie['duracion']}",
        f"  Genero: {movie['genero']}",
        f"  Sinopsis: {movie['sinopsis']}",
        f"  URL: {movie['url']}",
    ]
    return "\n".join(lines)


# Registramos como modulo para que cartelera_scraper / telegram_bot puedan importarlo
movie_scraper = types.ModuleType("movie_scraper")
movie_scraper.get_movie_info = get_movie_info
movie_scraper.search_movie = search_movie
movie_scraper.scrape_movie_page = scrape_movie_page
movie_scraper.load_cache = load_cache
movie_scraper.save_cache = save_cache
movie_scraper.format_movie_text = format_movie_text
movie_scraper.CAMPOS_VALIDOS = CAMPOS_VALIDOS
sys.modules["movie_scraper"] = movie_scraper

print("movie_scraper cargado.")

### Prueba rapida del scraper de SensaCine

In [ ]:
info = get_movie_info("Inception")
if info:
    print(format_movie_text(info))
else:
    print("No encontrada")

## 5. Scraper de cartelera de Madrid (`cartelera_scraper.py`)

Scrapea los principales cines de Madrid en eCartelera, deduplica por titulo, enriquece con datos de SensaCine, aplica el filtro por perfil del usuario y permite envio por Telegram.

In [ ]:
# cartelera_scraper.py
import argparse
import json
import os
import sys
import re
import time
import types
import requests
from bs4 import BeautifulSoup

import config
from movie_scraper import get_movie_info

# ============================================================
# Cines de Madrid en ecartelera.com
# ============================================================

CINES_MADRID = [
    ("Yelmo Cines Ideal", "https://www.ecartelera.com/cines/54,0,1.html"),
    ("Callao", "https://www.ecartelera.com/cines/8,0,1.html"),
    ("Cinesa Proyecciones", "https://www.ecartelera.com/cines/17,0,1.html"),
    ("Cines Princesa", "https://www.ecartelera.com/cines/20,0,1.html"),
    ("Palacio de la Prensa", "https://www.ecartelera.com/cines/38,0,1.html"),
    ("Renoir Plaza de España", "https://www.ecartelera.com/cines/44,0,1.html"),
    ("Cinesa Príncipe Pío", "https://www.ecartelera.com/cines/53,0,1.html"),
]


def scrape_cinema(cinema_name, cinema_url):
    """Scrapea las peliculas en cartelera de un cine concreto."""
    try:
        r = requests.get(cinema_url, headers=config.REQUEST_HEADERS, timeout=15)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"  Error al acceder a {cinema_name}: {e}", file=sys.stderr)
        return []

    soup = BeautifulSoup(r.text, "lxml")
    items = soup.find_all("div", class_="titem")
    movies = []

    for item in items:
        title_el = item.find("p", class_="tit")
        if not title_el:
            continue

        link_el = title_el.find("a")
        title = link_el.get_text(strip=True) if link_el else title_el.get_text(strip=True)
        ecartelera_url = link_el.get("href", "") if link_el else ""

        data_el = item.find("p", class_="data")
        data_spans = data_el.find_all("span") if data_el else []
        duracion = data_spans[0].get_text(strip=True) if len(data_spans) > 0 else ""
        pais = data_spans[1].get_text(strip=True) if len(data_spans) > 1 else ""
        genero = data_spans[2].get_text(strip=True) if len(data_spans) > 2 else ""
        clasificacion = data_spans[3].get_text(strip=True) if len(data_spans) > 3 else ""

        dir_el = item.find("p", class_="dir")
        director = ""
        if dir_el:
            dir_links = dir_el.find_all("a")
            director = ", ".join(a.get_text(strip=True) for a in dir_links)

        score_el = item.find("span", class_="nota")
        nota_ecartelera = score_el.get_text(strip=True) if score_el else ""

        sessions_el = item.find("div", class_="sessions")
        horarios = []
        if sessions_el:
            for li in sessions_el.find_all("li"):
                session = li.find(["a", "span"], attrs={"data-session-time": True})
                if session:
                    horarios.append(session.get("data-session-time", ""))

        movies.append({
            "titulo": title,
            "duracion": duracion,
            "pais": pais,
            "genero": genero,
            "clasificacion": clasificacion,
            "director": director,
            "nota_ecartelera": nota_ecartelera,
            "horarios": horarios,
            "ecartelera_url": ecartelera_url,
            "cine": cinema_name,
        })

    return movies


def get_cartelera_madrid():
    """Obtiene la cartelera completa de Madrid (deduplicada por titulo)."""
    all_movies = {}

    for cinema_name, cinema_url in CINES_MADRID:
        print(f"  Scrapeando {cinema_name}...", file=sys.stderr)
        movies = scrape_cinema(cinema_name, cinema_url)

        for m in movies:
            key = m["titulo"].lower().strip()
            if key not in all_movies:
                all_movies[key] = {
                    "titulo": m["titulo"],
                    "duracion": m["duracion"],
                    "pais": m["pais"],
                    "genero": m["genero"],
                    "director": m["director"],
                    "nota_ecartelera": m["nota_ecartelera"],
                    "ecartelera_url": m["ecartelera_url"],
                    "cines": {},
                }

            cine_name = m["cine"]
            if cine_name not in all_movies[key]["cines"]:
                all_movies[key]["cines"][cine_name] = m["horarios"]
            else:
                all_movies[key]["cines"][cine_name].extend(m["horarios"])

            if not all_movies[key]["nota_ecartelera"] and m["nota_ecartelera"]:
                all_movies[key]["nota_ecartelera"] = m["nota_ecartelera"]

        time.sleep(0.5)

    return list(all_movies.values())


def enrich_with_sensacine(movies):
    """Enriquece las peliculas de cartelera con datos de SensaCine."""
    enriched = []
    for m in movies:
        print(f"  Buscando en SensaCine: {m['titulo']}...", file=sys.stderr)
        sc_info = get_movie_info(m["titulo"])

        if sc_info:
            m["nota_sensacine"] = sc_info.get("nota", "N/A")
            m["nota_escala"] = sc_info.get("nota_escala", "/5")
            m["votos_sensacine"] = sc_info.get("votos", 0)
            m["sinopsis"] = sc_info.get("sinopsis", "")
            m["genero_sensacine"] = sc_info.get("genero", "")
            m["sensacine_url"] = sc_info.get("url", "")
            m["poster"] = sc_info.get("poster", "")
        else:
            m["nota_sensacine"] = "N/A"
            m["nota_escala"] = "/5"
            m["votos_sensacine"] = 0
            m["sinopsis"] = ""
            m["genero_sensacine"] = m.get("genero", "")
            m["sensacine_url"] = ""
            m["poster"] = ""

        enriched.append(m)
        time.sleep(0.3)

    return enriched


def load_user_profile():
    """Carga el perfil de usuario desde disco."""
    path = config.USER_PROFILE_FILE
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"genres": {}, "favorite_directors": []}


def filter_by_profile(movies, profile=None):
    """Filtra peliculas segun el perfil del usuario (nota minima por genero + directores favoritos)."""
    if profile is None:
        profile = load_user_profile()

    genre_filters = profile.get("genres", {})
    fav_directors = [d.lower() for d in profile.get("favorite_directors", [])]

    if not genre_filters and not fav_directors:
        return movies

    filtered = []
    for m in movies:
        director = m.get("director", "").lower()
        if any(fav in director for fav in fav_directors):
            filtered.append(m)
            continue

        nota = m.get("nota_sensacine", "N/A")
        if nota == "N/A":
            continue

        nota = float(nota)
        genres_movie = m.get("genero_sensacine", "") or m.get("genero", "")

        passed = False
        for genre, min_nota in genre_filters.items():
            if genre.lower() in genres_movie.lower():
                if nota >= min_nota:
                    passed = True
                    break

        if not passed and not any(g.lower() in genres_movie.lower() for g in genre_filters):
            if nota >= 3.5:
                passed = True

        if passed:
            filtered.append(m)

    return filtered


def send_telegram(message, chat_id=None):
    """Envia un mensaje por Telegram (admite chunking para mensajes >4096 chars)."""
    token = config.TELEGRAM_BOT_TOKEN
    chat_id = chat_id or config.TELEGRAM_CHAT_ID

    if not token or token == "TU_TOKEN_AQUI" or not chat_id:
        print("Error: Configura TELEGRAM_BOT_TOKEN y TELEGRAM_CHAT_ID en config.py",
              file=sys.stderr)
        return False

    url = f"https://api.telegram.org/bot{token}/sendMessage"
    chunks = [message[i:i+4000] for i in range(0, len(message), 4000)]

    for chunk in chunks:
        payload = {
            "chat_id": chat_id,
            "text": chunk,
            "parse_mode": "HTML",
            "disable_web_page_preview": True,
        }
        try:
            r = requests.post(url, json=payload, timeout=15)
            r.raise_for_status()
        except requests.RequestException as e:
            print(f"Error enviando por Telegram: {e}", file=sys.stderr)
            return False

    return True


def format_cartelera_text(movies):
    """Formatea la cartelera como texto legible."""
    lines = ["=" * 55]
    lines.append("  CARTELERA DE CINE - MADRID")
    lines.append("=" * 55)

    for m in movies:
        nota_sc = m.get("nota_sensacine", "N/A")
        escala = m.get("nota_escala", "/5")
        nota_str = f"{nota_sc}{escala}" if nota_sc != "N/A" else "Sin nota"
        lines.append(f"\n  {m['titulo']}")
        lines.append(f"  {'─' * 40}")
        if nota_sc != "N/A":
            lines.append(f"  Nota SensaCine: {nota_str}")
        if m.get("nota_ecartelera"):
            lines.append(f"  Nota eCartelera: {m['nota_ecartelera']}")
        if m.get("genero_sensacine"):
            lines.append(f"  Genero: {m['genero_sensacine']}")
        elif m.get("genero"):
            lines.append(f"  Genero: {m['genero']}")
        if m.get("director"):
            lines.append(f"  Director: {m['director']}")
        if m.get("duracion"):
            lines.append(f"  Duracion: {m['duracion']}")
        if m.get("sinopsis"):
            lines.append(f"  Sinopsis: {m['sinopsis'][:150]}...")

        cines = m.get("cines", {})
        if cines:
            lines.append(f"  Cines ({len(cines)}):")
            for cine, horarios in cines.items():
                horarios_str = ", ".join(horarios) if horarios else "consultar"
                lines.append(f"    - {cine}: {horarios_str}")

        if m.get("ecartelera_url"):
            lines.append(f"  eCartelera: {m['ecartelera_url']}")
        if m.get("sensacine_url"):
            lines.append(f"  SensaCine: {m['sensacine_url']}")

    lines.append(f"\n{'=' * 55}")
    lines.append(f"  Total: {len(movies)} peliculas")
    lines.append("=" * 55)
    return "\n".join(lines)


def format_cartelera_telegram(movies):
    """Formatea la cartelera para Telegram (HTML)."""
    lines = ["<b>🎬 CARTELERA DE CINE - MADRID</b>\n"]

    for m in movies:
        nota_sc = m.get("nota_sensacine", "N/A")
        escala = m.get("nota_escala", "/5")
        nota_str = f"{nota_sc}{escala}" if nota_sc != "N/A" else "Sin nota"
        title = m["titulo"]

        lines.append(f"<b>{title}</b>")
        lines.append(f"⭐ Nota SensaCine: {nota_str}")
        if m.get("genero_sensacine"):
            lines.append(f"🎭 {m['genero_sensacine']}")
        if m.get("director"):
            lines.append(f"🎬 Dir: {m['director']}")

        links = []
        if m.get("ecartelera_url"):
            links.append(f'<a href="{m["ecartelera_url"]}">Ficha eCartelera</a>')
        if m.get("sensacine_url"):
            links.append(f'<a href="{m["sensacine_url"]}">Ficha SensaCine</a>')
        if links:
            lines.append("🔗 " + " | ".join(links))

        lines.append("─" * 30)

    lines.append(f"\n<b>Total: {len(movies)} peliculas</b>")
    return "\n".join(lines)


# Registramos como modulo
cartelera_scraper = types.ModuleType("cartelera_scraper")
cartelera_scraper.CINES_MADRID = CINES_MADRID
cartelera_scraper.scrape_cinema = scrape_cinema
cartelera_scraper.get_cartelera_madrid = get_cartelera_madrid
cartelera_scraper.enrich_with_sensacine = enrich_with_sensacine
cartelera_scraper.load_user_profile = load_user_profile
cartelera_scraper.filter_by_profile = filter_by_profile
cartelera_scraper.send_telegram = send_telegram
cartelera_scraper.format_cartelera_text = format_cartelera_text
cartelera_scraper.format_cartelera_telegram = format_cartelera_telegram
sys.modules["cartelera_scraper"] = cartelera_scraper

print("cartelera_scraper cargado.")

### Prueba: cartelera filtrada por perfil

In [ ]:
movies = get_cartelera_madrid()
movies = enrich_with_sensacine(movies)
movies_filtradas = filter_by_profile(movies)
print(format_cartelera_text(movies_filtradas))

## 6. Bot de Telegram (`telegram_bot.py`)

Bot conversacional con comandos `/pelicula`, `/nota`, `/director`, `/duracion`, `/sinopsis`, `/cartelera`, `/perfil`. Ademas integra un **LLM open source via Ollama** (qwen2.5:3b) para generar comentarios naturales sobre cada pelicula.

> Esta celda **no se ejecuta automaticamente** porque `app.run_polling()` bloquea el kernel. Para arrancar el bot, llama manualmente a `bot_main()` desde una celda dedicada.

In [ ]:
# telegram_bot.py
import json
import logging
import sys

try:
    from telegram import Update
    from telegram.ext import (
        Application,
        CommandHandler,
        MessageHandler,
        ContextTypes,
        filters,
    )
    TELEGRAM_AVAILABLE = True
except ImportError:
    print("AVISO: python-telegram-bot no instalado. Instalar con: pip install python-telegram-bot",
          file=sys.stderr)
    TELEGRAM_AVAILABLE = False

import config
from movie_scraper import get_movie_info
from cartelera_scraper import (
    get_cartelera_madrid,
    enrich_with_sensacine,
    filter_by_profile,
    load_user_profile,
)

logging.basicConfig(
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)


# ============================================================
# Integracion con LLM Open Source (Ollama)
# ============================================================

def query_llm(prompt):
    """Consulta a Ollama. Si no esta disponible devuelve None."""
    try:
        import ollama
        response = ollama.chat(
            model=config.OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": "Eres un asistente experto en cine. Responde de forma concisa y amigable en español."},
                {"role": "user", "content": prompt},
            ],
        )
        return response["message"]["content"]
    except Exception:
        return None


def format_movie_response(info, field=None):
    """Formatea la respuesta de una pelicula para Telegram."""
    escala = info.get("nota_escala", "/5")
    if field == "nota":
        text = f"⭐ <b>{info['titulo']}</b> ({info['año']})\nNota SensaCine: {info['nota']}{escala} ({info['votos']:,} votos)"
    elif field == "director":
        text = f"🎬 <b>{info['titulo']}</b> ({info['año']})\nDirector: {info['director']}"
    elif field == "duracion":
        text = f"⏱ <b>{info['titulo']}</b> ({info['año']})\nDuración: {info['duracion']}"
    elif field == "sinopsis":
        text = f"📖 <b>{info['titulo']}</b> ({info['año']})\n\n{info['sinopsis']}"
    else:
        text = (
            f"🎬 <b>{info['titulo']}</b> ({info['año']})\n"
            f"{'─' * 25}\n"
            f"⭐ Nota: {info['nota']}{escala} ({info['votos']:,} votos)\n"
            f"🎭 Género: {info['genero']}\n"
            f"👤 Director: {info['director']}\n"
            f"⏱ Duración: {info['duracion']}\n"
            f"📖 Sinopsis: {info['sinopsis']}\n"
            f"\n🔗 <a href=\"{info['url']}\">Ver en SensaCine</a>"
        )
    return text


if TELEGRAM_AVAILABLE:
    # ============================================================
    # Handlers de comandos
    # ============================================================

    async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
        await update.message.reply_text(
            "🎬 <b>Agente de Peliculas</b>\n\n"
            "Puedo darte informacion sobre cualquier pelicula.\n\n"
            "<b>Comandos:</b>\n"
            "/pelicula &lt;nombre&gt; - Info completa\n"
            "/nota &lt;nombre&gt; - Nota SensaCine\n"
            "/director &lt;nombre&gt; - Director\n"
            "/duracion &lt;nombre&gt; - Duracion\n"
            "/sinopsis &lt;nombre&gt; - Sinopsis\n"
            "/cartelera - Cartelera de Madrid\n"
            "/perfil - Ver perfil de filtrado\n"
            "/ayuda - Ayuda\n\n"
            "O simplemente escribe el nombre de una pelicula.",
            parse_mode="HTML",
        )

    async def ayuda(update: Update, context: ContextTypes.DEFAULT_TYPE):
        await start(update, context)

    async def pelicula_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
        if not context.args:
            await update.message.reply_text("Uso: /pelicula <nombre de la pelicula>")
            return
        movie_name = " ".join(context.args)
        await update.message.reply_text(f"🔍 Buscando '{movie_name}'...")
        info = get_movie_info(movie_name)
        if info:
            response = format_movie_response(info)
            escala = info.get("nota_escala", "/5")
            llm_response = query_llm(
                f"En una frase breve, recomienda o comenta sobre la pelicula '{info['titulo']}' "
                f"({info['año']}) dirigida por {info['director']}. Nota SensaCine: {info['nota']}{escala}."
            )
            if llm_response:
                response += f"\n\n🤖 <i>{llm_response}</i>"
            await update.message.reply_text(response, parse_mode="HTML", disable_web_page_preview=True)
        else:
            await update.message.reply_text(f"❌ No encontré la pelicula '{movie_name}'.")

    async def nota_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
        if not context.args:
            await update.message.reply_text("Uso: /nota <nombre de la pelicula>")
            return
        movie_name = " ".join(context.args)
        info = get_movie_info(movie_name)
        if info:
            await update.message.reply_text(format_movie_response(info, "nota"), parse_mode="HTML")
        else:
            await update.message.reply_text(f"❌ No encontré '{movie_name}'.")

    async def director_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
        if not context.args:
            await update.message.reply_text("Uso: /director <nombre de la pelicula>")
            return
        movie_name = " ".join(context.args)
        info = get_movie_info(movie_name)
        if info:
            await update.message.reply_text(format_movie_response(info, "director"), parse_mode="HTML")
        else:
            await update.message.reply_text(f"❌ No encontré '{movie_name}'.")

    async def duracion_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
        if not context.args:
            await update.message.reply_text("Uso: /duracion <nombre de la pelicula>")
            return
        movie_name = " ".join(context.args)
        info = get_movie_info(movie_name)
        if info:
            await update.message.reply_text(format_movie_response(info, "duracion"), parse_mode="HTML")
        else:
            await update.message.reply_text(f"❌ No encontré '{movie_name}'.")

    async def sinopsis_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
        if not context.args:
            await update.message.reply_text("Uso: /sinopsis <nombre de la pelicula>")
            return
        movie_name = " ".join(context.args)
        info = get_movie_info(movie_name)
        if info:
            await update.message.reply_text(format_movie_response(info, "sinopsis"), parse_mode="HTML")
        else:
            await update.message.reply_text(f"❌ No encontré '{movie_name}'.")

    async def cartelera_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
        await update.message.reply_text("🎬 Obteniendo cartelera de Madrid... (puede tardar unos segundos)")
        movies = get_cartelera_madrid()
        movies = enrich_with_sensacine(movies)
        if context.args and "filtrar" in context.args:
            profile = load_user_profile()
            movies = filter_by_profile(movies, profile)

        def sort_key(m):
            nota = m.get("nota_sensacine", "N/A")
            return float(nota) if nota != "N/A" else 0
        movies.sort(key=sort_key, reverse=True)

        if not movies:
            await update.message.reply_text("No se encontraron peliculas en cartelera.")
            return

        lines = ["🎬 <b>CARTELERA DE CINE - MADRID</b>\n"]
        for m in movies[:15]:
            nota_sc = m.get("nota_sensacine", "N/A")
            escala = m.get("nota_escala", "/5")
            nota_str = f"{nota_sc}{escala}" if nota_sc != "N/A" else "Sin nota"
            title = m["titulo"]
            lines.append(f"<b>{title}</b>")
            lines.append(f"⭐ {nota_str}")
            if m.get("genero_sensacine"):
                lines.append(f"🎭 {m['genero_sensacine']}")
            links = []
            if m.get("ecartelera_url"):
                links.append(f'<a href="{m["ecartelera_url"]}">eCartelera</a>')
            if m.get("sensacine_url"):
                links.append(f'<a href="{m["sensacine_url"]}">SensaCine</a>')
            if links:
                lines.append("🔗 " + " | ".join(links))
            lines.append("")
        lines.append(f"<b>Total: {len(movies)} peliculas</b>")
        msg = "\n".join(lines)
        await update.message.reply_text(msg, parse_mode="HTML", disable_web_page_preview=True)

    async def perfil_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
        profile = load_user_profile()
        genres = profile.get("genres", {})
        directors = profile.get("favorite_directors", [])
        text = "👤 <b>Perfil de Usuario</b>\n\n"
        text += "<b>Géneros (nota mínima):</b>\n"
        for genre, min_nota in genres.items():
            text += f"  • {genre}: {min_nota}\n"
        text += f"\n<b>Directores favoritos:</b>\n"
        for d in directors:
            text += f"  • {d}\n"
        text += "\n<i>Edita user_profile.json para cambiar preferencias.</i>"
        await update.message.reply_text(text, parse_mode="HTML")

    async def text_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
        movie_name = update.message.text.strip()
        if not movie_name:
            return
        info = get_movie_info(movie_name)
        if info:
            response = format_movie_response(info)
            escala = info.get("nota_escala", "/5")
            llm_response = query_llm(
                f"En una frase breve, recomienda o comenta sobre la pelicula '{info['titulo']}' "
                f"({info['año']}) dirigida por {info['director']}. Nota SensaCine: {info['nota']}{escala}."
            )
            if llm_response:
                response += f"\n\n🤖 <i>{llm_response}</i>"
            await update.message.reply_text(response, parse_mode="HTML", disable_web_page_preview=True)
        else:
            await update.message.reply_text(
                f"❌ No encontré '{movie_name}'.\n"
                "Prueba con /ayuda para ver los comandos disponibles."
            )

    def bot_main():
        """Arranca el bot de Telegram (bloquea el kernel mientras este activo)."""
        token = config.TELEGRAM_BOT_TOKEN
        if not token or token == "TU_TOKEN_AQUI":
            print("Error: Configura TELEGRAM_BOT_TOKEN en config.py", file=sys.stderr)
            return

        app = Application.builder().token(token).build()
        app.add_handler(CommandHandler("start", start))
        app.add_handler(CommandHandler("ayuda", ayuda))
        app.add_handler(CommandHandler("help", ayuda))
        app.add_handler(CommandHandler("pelicula", pelicula_cmd))
        app.add_handler(CommandHandler("nota", nota_cmd))
        app.add_handler(CommandHandler("director", director_cmd))
        app.add_handler(CommandHandler("duracion", duracion_cmd))
        app.add_handler(CommandHandler("sinopsis", sinopsis_cmd))
        app.add_handler(CommandHandler("cartelera", cartelera_cmd))
        app.add_handler(CommandHandler("perfil", perfil_cmd))
        app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, text_message))
        print("Bot de Telegram iniciado. Presiona Ctrl+C para detener.")
        app.run_polling()

    print("Handlers de Telegram listos. Llama a bot_main() para arrancar el bot.")
else:
    print("python-telegram-bot no disponible: omitiendo handlers.")

## 7. Interfaz web Flask (`web_app.py`)

Servidor Flask con rutas `/`, `/buscar`, `/cartelera`, `/api/pelicula/<nombre>`, `/api/cartelera`. Usa `templates/index.html` (no incluido en el notebook por ser HTML estatico).

In [ ]:
# web_app.py
import json
import sys

try:
    from flask import Flask, render_template, request, jsonify
    FLASK_AVAILABLE = True
except ImportError:
    print("Flask no instalado. Instalar con: pip install flask", file=sys.stderr)
    FLASK_AVAILABLE = False

import config
from movie_scraper import get_movie_info
from cartelera_scraper import (
    get_cartelera_madrid,
    enrich_with_sensacine,
    filter_by_profile,
    load_user_profile,
)

if FLASK_AVAILABLE:
    app = Flask(__name__)

    @app.route("/")
    def index():
        return render_template("index.html")

    @app.route("/buscar", methods=["POST"])
    def buscar():
        movie_name = request.form.get("pelicula", "").strip()
        campo = request.form.get("campo", "")
        if not movie_name:
            return render_template("index.html", error="Introduce el nombre de una pelicula.")
        info = get_movie_info(movie_name)
        if not info:
            return render_template("index.html", error=f"No se encontro: {movie_name}")
        return render_template("index.html", movie=info, campo=campo, query=movie_name)

    @app.route("/api/pelicula/<nombre>")
    def api_pelicula(nombre):
        info = get_movie_info(nombre)
        if not info:
            return jsonify({"error": f"No se encontro: {nombre}"}), 404
        return jsonify(info)

    @app.route("/cartelera")
    def cartelera():
        filtrar = request.args.get("filtrar", "false") == "true"
        movies = get_cartelera_madrid()
        movies = enrich_with_sensacine(movies)
        if filtrar:
            profile = load_user_profile()
            movies = filter_by_profile(movies, profile)

        def sort_key(m):
            nota = m.get("nota_sensacine", "N/A")
            return float(nota) if nota != "N/A" else 0
        movies.sort(key=sort_key, reverse=True)

        for m in movies:
            if "cines" in m and isinstance(m["cines"], dict):
                m["cines"] = {k: list(v) for k, v in m["cines"].items()}

        return render_template("index.html", cartelera=movies, filtrar=filtrar)

    @app.route("/api/cartelera")
    def api_cartelera():
        movies = get_cartelera_madrid()
        movies = enrich_with_sensacine(movies)
        for m in movies:
            if "cines" in m and isinstance(m["cines"], dict):
                m["cines"] = {k: list(v) for k, v in m["cines"].items()}

        def sort_key(m):
            nota = m.get("nota_sensacine", "N/A")
            return float(nota) if nota != "N/A" else 0
        movies.sort(key=sort_key, reverse=True)
        return jsonify(movies)

    def web_main():
        """Arranca el servidor Flask (bloqueante)."""
        app.run(host=config.FLASK_HOST, port=config.FLASK_PORT, debug=config.FLASK_DEBUG)

    print("Flask app lista. Llama a web_main() para arrancarla.")
else:
    print("Flask no disponible: omitiendo definicion de la app.")

## 8. Skill de Alexa (`alexa_lambda.py`)

Lambda con los siguientes intents:
- `GetRatingIntent`, `GetDirectorIntent`, `GetDurationIntent`, `GetSynopsisIntent`,
  `GetVotesIntent`, `GetGenreIntent`, `GetAllInfoIntent`
- `AMAZON.HelpIntent`, `AMAZON.CancelIntent`, `AMAZON.StopIntent`

Despliegue: subir este codigo + `movie_scraper.py` + `config.py` a AWS Lambda o usar Alexa-hosted skill.

In [ ]:
# alexa_lambda.py
import json
import os
import sys

try:
    from ask_sdk_core.skill_builder import SkillBuilder
    from ask_sdk_core.dispatch_components import (
        AbstractRequestHandler,
        AbstractExceptionHandler,
    )
    from ask_sdk_core.utils import is_request_type, is_intent_name
    from ask_sdk_model.ui import SimpleCard
    ASK_AVAILABLE = True
except ImportError:
    print("AVISO: ask-sdk-core no instalado. Instalar con: pip install ask-sdk-core",
          file=sys.stderr)
    ASK_AVAILABLE = False

from movie_scraper import get_movie_info

movie_cache = {}

def get_cached_movie(title):
    """Cache en memoria + disco."""
    key = title.lower().strip()
    if key in movie_cache:
        return movie_cache[key]
    info = get_movie_info(title)
    if info:
        movie_cache[key] = info
    return info


if ASK_AVAILABLE:
    class LaunchRequestHandler(AbstractRequestHandler):
        def can_handle(self, handler_input):
            return is_request_type("LaunchRequest")(handler_input)

        def handle(self, handler_input):
            speech = ("Bienvenido al agente de peliculas. "
                      "Puedes preguntarme sobre cualquier pelicula. "
                      "Por ejemplo, di: cual es la nota de Inception.")
            return (
                handler_input.response_builder
                .speak(speech)
                .ask("¿Sobre que pelicula quieres saber?")
                .set_card(SimpleCard("Agente de Peliculas", speech))
                .response
            )

    class GetRatingIntentHandler(AbstractRequestHandler):
        def can_handle(self, handler_input):
            return is_intent_name("GetRatingIntent")(handler_input)

        def handle(self, handler_input):
            movie_name = handler_input.request_envelope.request.intent.slots["movie"].value
            info = get_cached_movie(movie_name)
            if info:
                nota = info.get("nota", "N/A")
                speech = f"La nota de {info['titulo']} en IMDB es {nota} sobre 10."
            else:
                speech = f"Lo siento, no he encontrado la pelicula {movie_name}."
            return (
                handler_input.response_builder
                .speak(speech)
                .set_card(SimpleCard("Nota", speech))
                .response
            )

    class GetDirectorIntentHandler(AbstractRequestHandler):
        def can_handle(self, handler_input):
            return is_intent_name("GetDirectorIntent")(handler_input)

        def handle(self, handler_input):
            movie_name = handler_input.request_envelope.request.intent.slots["movie"].value
            info = get_cached_movie(movie_name)
            if info:
                director = info.get("director", "desconocido")
                speech = f"{info['titulo']} fue dirigida por {director}."
            else:
                speech = f"Lo siento, no he encontrado la pelicula {movie_name}."
            return (
                handler_input.response_builder
                .speak(speech)
                .set_card(SimpleCard("Director", speech))
                .response
            )

    class GetDurationIntentHandler(AbstractRequestHandler):
        def can_handle(self, handler_input):
            return is_intent_name("GetDurationIntent")(handler_input)

        def handle(self, handler_input):
            movie_name = handler_input.request_envelope.request.intent.slots["movie"].value
            info = get_cached_movie(movie_name)
            if info:
                duracion = info.get("duracion", "desconocida")
                speech = f"{info['titulo']} dura {duracion}."
            else:
                speech = f"Lo siento, no he encontrado la pelicula {movie_name}."
            return (
                handler_input.response_builder
                .speak(speech)
                .set_card(SimpleCard("Duracion", speech))
                .response
            )

    class GetSynopsisIntentHandler(AbstractRequestHandler):
        def can_handle(self, handler_input):
            return is_intent_name("GetSynopsisIntent")(handler_input)

        def handle(self, handler_input):
            movie_name = handler_input.request_envelope.request.intent.slots["movie"].value
            info = get_cached_movie(movie_name)
            if info:
                sinopsis = info.get("sinopsis", "No disponible")
                speech = f"La sinopsis de {info['titulo']} es: {sinopsis}"
            else:
                speech = f"Lo siento, no he encontrado la pelicula {movie_name}."
            return (
                handler_input.response_builder
                .speak(speech)
                .set_card(SimpleCard("Sinopsis", speech))
                .response
            )

    class GetVotesIntentHandler(AbstractRequestHandler):
        def can_handle(self, handler_input):
            return is_intent_name("GetVotesIntent")(handler_input)

        def handle(self, handler_input):
            movie_name = handler_input.request_envelope.request.intent.slots["movie"].value
            info = get_cached_movie(movie_name)
            if info:
                votos = info.get("votos", 0)
                speech = f"{info['titulo']} tiene {votos:,} votos en IMDB."
            else:
                speech = f"Lo siento, no he encontrado la pelicula {movie_name}."
            return (
                handler_input.response_builder
                .speak(speech)
                .set_card(SimpleCard("Votos", speech))
                .response
            )

    class GetGenreIntentHandler(AbstractRequestHandler):
        def can_handle(self, handler_input):
            return is_intent_name("GetGenreIntent")(handler_input)

        def handle(self, handler_input):
            movie_name = handler_input.request_envelope.request.intent.slots["movie"].value
            info = get_cached_movie(movie_name)
            if info:
                genero = info.get("genero", "desconocido")
                speech = f"El genero de {info['titulo']} es {genero}."
            else:
                speech = f"Lo siento, no he encontrado la pelicula {movie_name}."
            return (
                handler_input.response_builder
                .speak(speech)
                .set_card(SimpleCard("Genero", speech))
                .response
            )

    class GetAllInfoIntentHandler(AbstractRequestHandler):
        def can_handle(self, handler_input):
            return is_intent_name("GetAllInfoIntent")(handler_input)

        def handle(self, handler_input):
            movie_name = handler_input.request_envelope.request.intent.slots["movie"].value
            info = get_cached_movie(movie_name)
            if info:
                speech = (
                    f"{info['titulo']}, del año {info.get('año', 'desconocido')}. "
                    f"Dirigida por {info.get('director', 'desconocido')}. "
                    f"Genero: {info.get('genero', 'desconocido')}. "
                    f"Duracion: {info.get('duracion', 'desconocida')}. "
                    f"Nota en IMDB: {info.get('nota', 'N/A')} sobre 10 "
                    f"con {info.get('votos', 0):,} votos. "
                    f"Sinopsis: {info.get('sinopsis', 'no disponible')}"
                )
            else:
                speech = f"Lo siento, no he encontrado la pelicula {movie_name}."
            return (
                handler_input.response_builder
                .speak(speech)
                .set_card(SimpleCard("Info Completa", speech))
                .response
            )

    class HelpIntentHandler(AbstractRequestHandler):
        def can_handle(self, handler_input):
            return is_intent_name("AMAZON.HelpIntent")(handler_input)

        def handle(self, handler_input):
            speech = ("Puedes preguntarme sobre cualquier pelicula. "
                      "Prueba con: cual es la nota de Inception, "
                      "quien dirigio The Matrix, "
                      "cuanto dura Interstellar, "
                      "o dime todo sobre Pulp Fiction.")
            return (
                handler_input.response_builder
                .speak(speech)
                .ask(speech)
                .response
            )

    class CancelAndStopIntentHandler(AbstractRequestHandler):
        def can_handle(self, handler_input):
            return (is_intent_name("AMAZON.CancelIntent")(handler_input) or
                    is_intent_name("AMAZON.StopIntent")(handler_input))

        def handle(self, handler_input):
            return (
                handler_input.response_builder
                .speak("Hasta luego. Disfruta del cine.")
                .response
            )

    class SessionEndedRequestHandler(AbstractRequestHandler):
        def can_handle(self, handler_input):
            return is_request_type("SessionEndedRequest")(handler_input)

        def handle(self, handler_input):
            return handler_input.response_builder.response

    class CatchAllExceptionHandler(AbstractExceptionHandler):
        def can_handle(self, handler_input, exception):
            return True

        def handle(self, handler_input, exception):
            print(f"Error: {exception}", file=sys.stderr)
            speech = "Lo siento, ha ocurrido un error. Intentalo de nuevo."
            return (
                handler_input.response_builder
                .speak(speech)
                .ask(speech)
                .response
            )

    sb = SkillBuilder()
    sb.add_request_handler(LaunchRequestHandler())
    sb.add_request_handler(GetRatingIntentHandler())
    sb.add_request_handler(GetDirectorIntentHandler())
    sb.add_request_handler(GetDurationIntentHandler())
    sb.add_request_handler(GetSynopsisIntentHandler())
    sb.add_request_handler(GetVotesIntentHandler())
    sb.add_request_handler(GetGenreIntentHandler())
    sb.add_request_handler(GetAllInfoIntentHandler())
    sb.add_request_handler(HelpIntentHandler())
    sb.add_request_handler(CancelAndStopIntentHandler())
    sb.add_request_handler(SessionEndedRequestHandler())
    sb.add_exception_handler(CatchAllExceptionHandler())

    handler = sb.lambda_handler()
    print("Skill de Alexa registrada. Entry point: handler")
else:
    print("ask-sdk-core no disponible: omitiendo skill de Alexa.")

## 9. Modelo de interaccion de la skill (`alexa_interaction_model.json`)

Definicion completa de utterances/intents que se pega en la pestaña *JSON Editor* de la Alexa Developer Console.

In [ ]:
# alexa_interaction_model.json
interaction_model = {
    "interactionModel": {
        "languageModel": {
            "invocationName": "agente de peliculas",
            "intents": [
                {"name": "GetRatingIntent",
                 "slots": [{"name": "movie", "type": "AMAZON.Movie"}],
                 "samples": ["cual es la nota de {movie}", "que nota tiene {movie}",
                              "puntuacion de {movie}", "que puntuacion tiene {movie}",
                              "como esta valorada {movie}", "valoracion de {movie}",
                              "rating de {movie}", "nota de {movie}",
                              "cuanto tiene {movie} en IMDB"]},
                {"name": "GetDirectorIntent",
                 "slots": [{"name": "movie", "type": "AMAZON.Movie"}],
                 "samples": ["quien dirigio {movie}", "quien es el director de {movie}",
                              "director de {movie}", "quien hizo {movie}",
                              "de quien es {movie}", "quien dirige {movie}",
                              "que director tiene {movie}"]},
                {"name": "GetDurationIntent",
                 "slots": [{"name": "movie", "type": "AMAZON.Movie"}],
                 "samples": ["cuanto dura {movie}", "que duracion tiene {movie}",
                              "duracion de {movie}", "cuanto tiempo dura {movie}",
                              "cuantas horas dura {movie}", "cuantos minutos dura {movie}",
                              "lo que dura {movie}"]},
                {"name": "GetSynopsisIntent",
                 "slots": [{"name": "movie", "type": "AMAZON.Movie"}],
                 "samples": ["de que va {movie}", "de que trata {movie}",
                              "sinopsis de {movie}", "argumento de {movie}",
                              "cual es la trama de {movie}", "cuentame de que va {movie}",
                              "sobre que trata {movie}", "resumen de {movie}"]},
                {"name": "GetVotesIntent",
                 "slots": [{"name": "movie", "type": "AMAZON.Movie"}],
                 "samples": ["cuantos votos tiene {movie}", "numero de votos de {movie}",
                              "votos de {movie}", "cuanta gente ha votado {movie}",
                              "cuantas valoraciones tiene {movie}"]},
                {"name": "GetGenreIntent",
                 "slots": [{"name": "movie", "type": "AMAZON.Movie"}],
                 "samples": ["que genero es {movie}", "de que genero es {movie}",
                              "genero de {movie}", "que tipo de pelicula es {movie}",
                              "a que genero pertenece {movie}"]},
                {"name": "GetAllInfoIntent",
                 "slots": [{"name": "movie", "type": "AMAZON.Movie"}],
                 "samples": ["dime todo sobre {movie}", "informacion de {movie}",
                              "cuentame sobre {movie}", "todo sobre {movie}",
                              "que sabes de {movie}", "dame informacion de {movie}",
                              "ficha de {movie}", "dime los datos de {movie}"]},
                {"name": "AMAZON.HelpIntent", "samples": []},
                {"name": "AMAZON.StopIntent", "samples": []},
                {"name": "AMAZON.CancelIntent", "samples": []},
                {"name": "AMAZON.FallbackIntent", "samples": []}
            ]
        }
    }
}

with open("alexa_interaction_model.json", "w", encoding="utf-8") as f:
    json.dump(interaction_model, f, ensure_ascii=False, indent=2)

print("alexa_interaction_model.json guardado.")

## 10. Automatizacion semanal (`cron_cartelera.sh`)

Script bash para invocar el scraper de cartelera con filtro y envio por Telegram. Se programa con `crontab`:

```
# Lunes a las 9:00
0 9 * * 1 /ruta/completa/cron_cartelera.sh
```

In [ ]:
# cron_cartelera.sh
cron_script = '''#!/bin/bash
# Script para ejecutar el scraper de cartelera via cron.
# Configurar en crontab con:
#   crontab -e
#   0 9 * * 1 /ruta/completa/cron_cartelera.sh
#
# Esto ejecuta el script todos los lunes a las 9:00.

SCRIPT_DIR="$(cd "$(dirname "$0")" && pwd)"
cd "$SCRIPT_DIR"

# Activar entorno virtual si existe
if [ -f "venv/bin/activate" ]; then
    source venv/bin/activate
fi

# Ejecutar scraper con filtro y envio por Telegram
python3 cartelera_scraper.py --filtrar --telegram >> cartelera_cron.log 2>&1

echo "[$(date)] Cartelera ejecutada" >> cartelera_cron.log
'''

with open("cron_cartelera.sh", "w", encoding="utf-8") as f:
    f.write(cron_script)

import os
os.chmod("cron_cartelera.sh", 0o755)
print("cron_cartelera.sh creado y marcado como ejecutable.")

## 11. Demo final

Pequeña demostracion de extremo a extremo: busqueda de una pelicula + cartelera filtrada.

In [ ]:
# Demo: info de una pelicula
demo_titles = ["The Matrix", "Interstellar", "Pulp Fiction"]
for t in demo_titles:
    info = get_movie_info(t)
    if info:
        print(f"\n=== {info['titulo']} ({info['año']}) ===")
        print(f"Nota: {info['nota']}{info['nota_escala']} | Director: {info['director']}")
        print(f"Genero: {info['genero']} | Duracion: {info['duracion']}")

In [ ]:
# Demo: cartelera filtrada por perfil + envio por Telegram (descomenta para enviar)
movies = get_cartelera_madrid()
movies = enrich_with_sensacine(movies)
movies_f = filter_by_profile(movies)

def sort_key(m):
    nota = m.get("nota_sensacine", "N/A")
    return float(nota) if nota != "N/A" else 0
movies_f.sort(key=sort_key, reverse=True)

print(format_cartelera_text(movies_f))

# Para enviar por Telegram:
# send_telegram(format_cartelera_telegram(movies_f))